### Objective

By the end of this notebook, we will have:

- Removed unnecessary columns
- Handled missing values
- Encoded categorical features
- Prepared the dataset for model training
- Saved the cleaned dataset

step 1: import the libraries 

In [94]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

import warnings
warnings.filterwarnings("ignore")

step 2: loading the dataset 

In [95]:
df = pd.read_csv(r"D:\machine learning\machine_learning\decsion_tree_project\ev__battery_life\dataset\raw_data\ev_battery_failure_dataset.csv")
df

,vehicle_id,vehicle_brand,vehicle_model,vehicle_type,manufacturing_year,battery_manufacturer,battery_chemistry,battery_capacity_kwh,drive_type,odometer_km,...,sensor_fault_count,BMS_warning_count,abnormal_voltage_events,battery_stress_index,aging_score,thermal_health_score,charging_quality_score,driving_stress_score,predicted_remaining_life_cycles,battery_failure
0,EV100000,Nissan,Leaf,SUV,2019.0,CATL,LMO,82.54,RWD,98163.0,...,2.0,3.0,0.0,0.0,53.5,66.0,65.2,0.0,401.0,0
1,EV100001,Audi,e-tron,Sedan,2018.0,Samsung SDI,NaN,64.17,AWD,113600.0,...,0.0,3.0,0.0,51.0,38.8,67.8,46.3,40.9,1094.0,0
2,EV100002,Toyota,bZ4X,Van,2018.0,Guoxuan,NMC,90.92,RWD,242253.0,...,4.0,1.0,0.0,36.2,66.1,48.6,0.0,6.5,645.0,0
3,EV100003,Volkswagen,ID.3,Hatchback,2022.0,CATL,NMC,47.35,AWD,83700.0,...,0.0,3.0,1.0,23.4,32.5,58.2,NaN,NaN,867.0,0
4,EV100004,Tesla,Model X,Crossover,2015.0,BYD Battery,NMC,68.04,FWD,88784.0,...,3.0,2.0,1.0,51.2,45.4,51.0,NaN,100.0,1090.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,EV299995,BYD,Song Plus,Hatchback,2016.0,CATL,LFP,45.99,FWD,182311.0,...,3.0,1.0,1.0,62.3,NaN,78.6,0.0,51.5,2691.0,0
199996,EV299996,Lucid,Air,Crossover,2015.0,LG Energy Solution,NMC,67.65,AWD,275317.0,...,2.0,5.0,2.0,33.1,89.5,59.3,73.7,57.0,420.0,0
199997,EV299997,BYD,Song Plus,SUV,NaN,BYD Battery,LFP,98.08,FWD,158747.0,...,2.0,2.0,0.0,79.0,13.1,64.7,32.8,7.6,3148.0,0
199998,EV299998,Tesla,Model Y,Hatchback,2016.0,SK On,NMC,51.49,FWD,95122.0,...,2.0,2.0,0.0,24.3,59.5,50.6,69.4,12.7,797.0,0


step 3: drop unnecessary columns
- these coulmns are unique identifiers and do not help the model learn meaningful patterns  

In [96]:
drop_columns = [
    "vehicle_id",
    "battery_serial"
]

df.drop(columns=drop_columns, inplace=True)

In [97]:
x = df.drop("battery_failure", axis=1)
y = df["battery_failure"]

In [98]:
df.shape

(200000, 68)

### Why Are We Dropping These Columns?


| Column         | Reason                              |
| -------------- | ----------------------------------- |
| vehicle_id     | Unique identifier for every vehicle |
| battery_serial | Unique serial number                |

If we keep them, the Decision Tree may memorize these values instead of learning general patterns, leading to overfitting.


step 4: separate feature and target 

In [99]:
print("Shape of x:", x.shape)
print("Shape of y:", y.shape)

Shape of x: (200000, 67)
Shape of y: (200000,)


step 5: identify numerical and categorical columns 

In [100]:
numerical_columns = x.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_columns = x.select_dtypes(include=["object"]).columns.tolist()

In [101]:
print("Numerical Features:", len(numerical_columns))
print("Categorical Features:", len(categorical_columns))

Numerical Features: 59
Categorical Features: 8


step 6: checking a missing values 

numerical

In [102]:
x[numerical_columns].isnull().sum().sort_values(ascending=False).head(10)

fast_charge_ratio         9879
average_speed             9849
manufacturing_year        9829
home_charging_ratio       9805
sensor_fault_count        9674
charging_interruptions    9648
state_of_charge           9636
state_of_health           9565
capacity_loss_percent     9559
slow_charge_ratio         9483
dtype: int64

categorical

In [103]:
x[categorical_columns].isnull().sum().sort_values(ascending=False)

fleet_or_private        8950
vehicle_model           8204
drive_type              7268
battery_chemistry       7214
terrain_type            7127
vehicle_brand           6436
vehicle_type            6432
battery_manufacturer    6188
dtype: int64

step 7: handle the missing values 
- numerical -> median 
- Median is preferred because it is less sensitive to outliers than the mean.

In [104]:
num_imputer = SimpleImputer(strategy="median")
x[numerical_columns] = num_imputer.fit_transform(
    x[numerical_columns]
)

Categorical → Most Frequent (Mode)

In [105]:
cat_imputer = SimpleImputer(strategy="most_frequent")

x[categorical_columns] = cat_imputer.fit_transform(
    x[categorical_columns]
)

Step 8: Verify Missing Values Again

In [106]:
print(x.isnull().sum().sum())

0


step 9: one - hot Encoding
- since we have only few categorical columns , one - hot encoding  is good choice 

In [107]:
encoder = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
encoded_features = encoder.fit_transform(x[categorical_columns])

step 10: convert Encoded features to DataFrame

In [108]:
encoded_df = pd.DataFrame(encoded_features, columns=encoder.get_feature_names_out(categorical_columns),index=x.index)

step 11: combine numerical and encoded features 
- remove the  original columns 

In [109]:
x = x.drop(columns=categorical_columns)

merge with the encoded features 

In [110]:
x = pd.concat([x, encoded_df], axis=1)

step 12: final dataset

In [111]:
print("Shape of x:", x.shape)

Shape of x: (200000, 156)


In [112]:
x.head()

,manufacturing_year,battery_capacity_kwh,odometer_km,vehicle_age_years,cycle_count,battery_health_percent,state_of_charge,depth_of_discharge,state_of_health,cell_voltage_avg,...,battery_chemistry_LTO,battery_chemistry_NCA,battery_chemistry_NMC,drive_type_FWD,drive_type_RWD,fleet_or_private_Private,terrain_type_Desert,terrain_type_Flat,terrain_type_Hilly,terrain_type_Mountainous
0,2019.0,82.54,98163.0,6.94,250.0,73.61,85.23,26.34,72.93,3.3558,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0
1,2018.0,64.17,113600.0,7.79,256.0,85.77,49.12,44.83,86.71,3.3658,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,2018.0,90.92,242253.0,7.85,556.0,72.37,54.94,33.65,72.17,3.4152,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
3,2022.0,47.35,83700.0,6.34,353.0,85.11,54.32,37.73,91.05,3.4202,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,2015.0,68.04,88784.0,11.13,287.0,87.40,62.73,65.29,86.17,3.4556,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0


step 13: combine features and target 

In [113]:
cleaned_df = pd.concat(
    [x, y],
    axis=1
)

step 14: save the dataset 

In [114]:
cleaned_df.to_csv(
    "D:/machine learning/machine_learning/decsion_tree_project/ev__battery_life/dataset/processed_data/cleaned_ev_battery_dataset.csv",
    index=False
)

## final verification 

In [115]:
cleaned_df.shape

(200000, 157)

## Summary

✔ Removed unnecessary identifier columns (`vehicle_id`, `battery_serial`).

✔ Separated features and target variable.

✔ Identified numerical and categorical features.

✔ Filled missing values:
- Numerical → Median
- Categorical → Most Frequent

✔ Applied One-Hot Encoding to categorical variables.

✔ Combined encoded and numerical features into a single dataset.

✔ Saved the cleaned dataset for model training.

### Next Notebook

**03_Exploratory_Data_Analysis.ipynb**

We'll explore:
- Class distribution
- Feature distributions
- Correlation analysis
- Outlier detection
- Relationships between key features and `battery_failure`
- Insights that can guide model development

In [116]:
(df["battery_failure"].value_counts(normalize=True)* 100).round(2)

battery_failure
0    90.04
1     9.96
Name: proportion, dtype: float64